# PID Loop Simulator
## Run a closed loop PID simulator to allow testing different options.
## This version extracts the open loop reproduction of data from a night to use for testing.
## I have also added multiple options for tracking the integral terms.
## This version now includes the option to control by Zernikes
Craig Lage 30-Apr-26

In [ ]:
import numpy as np
import pickle as pkl
import pandas as pd
import galsim
from collections import deque
import copy
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from lsst.ts.ofc import OFC, OFCData, StateEstimator, SensitivityMatrix

In [ ]:
class PIDLoopSimulation:
    """
    Closed-loop PID simulator that uses an open-loop reproduction (OLR)
    of nightly Zernike data to test different PID control strategies for
    the Rubin Observatory active optics system.
    """

    def __init__(
        self,
        ofc,
        ofc_data,
        state_estimator,
        sens_mat,
        kp,
        ki,
        kd,
        day_obs=20260420,
        seq_start=97,
        seq_end=147,
        n_correct=3,
        discard_intermediates=False,
        smith_corrector=True,
        control_vmodes=False,
        use_ofc_integral=True,
        max_integral=np.zeros(50),
        control_zernikes=False,
        n_integrals=10
    ):
        """
        Initialise the simulation and set all configuration options.

        Parameters
        ----------
        ofc : OFC
            Optical Feedback Controller instance (already configured).
        ofc_data : OFCData
            OFC data / configuration object.
        state_estimator : StateEstimator
            State estimator used for vmode calculations.
        sens_mat : np.ndarray
            Sensitivity matrix (shape: [n_sensors * n_zernikes, n_dof]).
        kp : np.ndarray, shape (50,)
            Proportional gain array for all 50 DOFs.
        ki : np.ndarray, shape (50,)
            Integral gain array for all 50 DOFs.
        kd : np.ndarray, shape (50,)
            Derivative gain array for all 50 DOFs.
        day_obs : int
            Observation day in YYYYMMDD format.
        seq_start : int
            First sequence number to include in the simulation.
        seq_end : int
            Last sequence number (exclusive) to include.
        n_correct : int
            Correction lag — how many exposures between applied corrections
            when discard_intermediates is True.
        discard_intermediates : bool
            If True, only apply a correction every n_correct steps.
            If False, apply corrections every step (optionally with Smith predictor).
        smith_corrector : bool
            When discard_intermediates is False, enable the Smith predictor
            correction to compensate for the control lag.
        control_vmodes : bool
            If True, operate in virtual-mode (vmode) space.
            If False, operate in degree-of-freedom (DOF) space.
        use_ofc_integral : bool
            If True, will use the integral term as coded in ofc.
            If False, operate in degree-of-freedom (DOF) space.
        max_integral : np.ndarray, shape (50,)
            Maximum absolute value the integral term is allowed to reach.
            Only applies when use_ofc_integral=True
        control_zernikes : bool
            If True, the control will be done in zernike space.
            If False, the control will be done in DoF space (or vmode space).
        n_integrals : int
            The number of integral terms to keep.
            Integral terms more than this are forgotten.
        """
        # ── External objects ──────────────────────────────────────────────
        self.ofc = ofc
        self.ofc_data = ofc_data
        self.state_estimator = state_estimator
        self.sens_mat = sens_mat

        # ── Data selection ────────────────────────────────────────────────
        self.day_obs = day_obs
        self.seq_start = seq_start
        self.seq_end = seq_end

        # ── Correction options ────────────────────────────────────────────
        self.n_correct = n_correct
        self.discard_intermediates = discard_intermediates
        self.smith_corrector = smith_corrector  # only used when discard_intermediates=False
        self.control_vmodes = control_vmodes

        # ── DOF index helpers ─────────────────────────────────────────────
        # Collapses the full 50-DOF space down to the 22 DOFs we are using.
        # If control_vmodes=True the _used arrays should have 12 components;
        # if control_vmodes=False they should have 22 components.
        self.indices = list(range(0, 17)) + list(range(30, 35))

        # ── PID gains and integrals (passed in from outside the class) ──
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.max_integral = max_integral
        self.use_ofc_integral = use_ofc_integral
        self.control_zernikes = control_zernikes
        self.n_integrals = n_integrals


        # ── Plot / label helpers ──────────────────────────────────────────
        self.zk_groups = [[0], [11 - 4], [1, 2], [3, 4], [5, 6], [22]]
        self.zk_group_labels = ['Z4', 'Z11', 'Z5 / Z6', 'Z7 / Z8', ' Z9 / Z10', 'Z22']

        self.groups = [[0], [5], [1, 2], [6, 7], [3, 4], [8, 9]]
        self.group_labels = [
            'M2 dz\n[um]', 'Cam dz\n[um]', 'M2 dx/dy\n[um]',
            'Cam dx/dy\n[um]', 'M2 tilts\n[arcsec]', 'Cam tilts\n[arcsec]',
        ]
        self.labels = [
            'm2 dz', 'm2 dx', 'm2 dy', 'm2 rx', 'm2 ry',
            'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
        ]

        self.mirror_groups = [
            [10, 11, 30, 31], [12, 34], [13, 14, 32, 33],
            [15, 16, 35, 36], [17, 18, 37, 38],
        ]
        self.mirror_group_labels = ['Astig', 'Spherical', 'Trefoil', 'Coma', 'Quad']

        self.all_labels = [
            'M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
            'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
            'B1,1', 'B1,2', 'B1,3', 'B1,4', 'B1,5',
            'B1,6', 'B1,7', 'B1,8', 'B1,9', 'B1,10',
            'B1,11', 'B1,12', 'B1,13', 'B1,14', 'B1,15',
            'B1,16', 'B1,17', 'B1,18', 'B1,19', 'B1,20',
            'B2,1', 'B2,2', 'B2,3', 'B2,4', 'B2,5',
            'B2,6', 'B2,7', 'B2,8', 'B2,9', 'B2,10',
            'B2,11', 'B2,12', 'B2,13', 'B2,14', 'B2,15',
            'B2,16', 'B2,17', 'B2,18', 'B2,19', 'B2,20',
        ]

        self.vmode_labels = [
            'Vmode1\nM2 tilts -rx-ry', 'Vmode2\nM2 tilts -rx+ry',
            'Vmode3\nCam tilts -rx+ry', 'Vmode4\nCam tilts rx+ry',
            'Vmode5\nZ4-Focus', 'Vmode6\nZ5-Astig-Oblique',
            'Vmode7\nZ6-Astig-Vert', 'Vmode8\nZ7-Coma-Vert',
            'Vmode9\nZ8-Coma-Horiz', 'Vmode10\nZ9-Trefoil-Vert',
            'Vmode11\nZ10-Trefoil-Oblique', 'Vmode12\nZ11-Spherical',
        ]

        # ── Simulation outputs (populated by runSimulation) ───────────────
        self.seqs = None
        self.z_measured = None
        self.z_opens = None
        self.rots = None
        self.filters = None
        self.new_zernikes = None
        self.integrals = None
        self.tweaks = None
        self.trims = None
        self.these_seqs = None

    # ─────────────────────────────────────────────────────────────────────
    # Helper methods
    # ─────────────────────────────────────────────────────────────────────

    def applyTrim(self, zernikes, trim, subtract=True):
        """
        Apply (or remove) a DOF trim vector to a set of Zernike measurements
        using the sensitivity matrix.

        Parameters
        ----------
        zernikes : np.ndarray, shape (4, 23)
            Measured Zernike coefficients at the four corner detectors.
        trim : np.ndarray, shape (22,)
            DOF trim vector (22 active DOFs).
        subtract : bool
            If True, subtract the trim contribution from the Zernikes
            (forward simulation step).
            If False, add it back (used to construct the open-loop reproduction).

        Returns
        -------
        np.ndarray, shape (4, 23)
            Zernike array with the trim contribution applied.
        """
        zernikes_change = self.sens_mat @ trim
        zernikes_change = zernikes_change.reshape(4, 21)
        # Zero out Z20 and Z21 (indices 16 and 17 after reshape)
        zernikes_change = np.insert(zernikes_change, obj=16, values=0, axis=1)
        zernikes_change = np.insert(zernikes_change, obj=16, values=0, axis=1)
        if subtract:
            return zernikes - zernikes_change
        else:
            # Removes the trim to reconstruct the open-loop Zernikes.
            return zernikes + zernikes_change

    def buildIntegralTerms(self):
        if self.use_ofc_integral:
            self.integral_terms = None
            return
        else:
            self.integral_terms = deque(maxlen=self.n_integrals)
            for i in range(self.n_integrals):
                if self.control_zernikes:
                    self.integral_terms.append(np.zeros([4,23]))
                else:
                    self.integral_terms.append(np.zeros(22))
            return
            
    def getTweak(self, zernikes, rotation_angle=0.0, filter_name='i',
                 subtract_intrinsics=False):
        """
        Compute the next OFC correction (tweak) for the active 22 DOFs.

        Parameters
        ----------
        zernikes : np.ndarray, shape (4, 23)
            Simulated Zernike coefficients at the four corner detectors.
        rotation_angle : float
            Instrument rotator angle in degrees.
        filter_name : str
            Photometric band name (e.g. 'i', 'r').
        subtract_intrinsics : bool
            If True, subtract the intrinsic Zernike offsets before computing
            the correction.

        Returns
        -------
        np.ndarray, shape (22,)
            Recommended DOF correction for the 22 active degrees of freedom.
        """
        sensor_ids = [191, 195, 199, 203]
        input_zernikes = copy.deepcopy(zernikes)
        if not self.use_ofc_integral and self.control_zernikes:
            integral_correction = self.ki * np.sum(self.integral_terms, axis=0)
            self.integral_terms.append(input_zernikes)
            input_zernikes = self.kp * input_zernikes + integral_correction
        self.ofc.calculate_corrections(
            input_zernikes, sensor_ids, filter_name,
            rotation_angle,
            subtract_intrinsics=subtract_intrinsics,
            control_vmodes=self.control_vmodes,
        )
        tweak = self.ofc.lv_dof[self.indices]
        if not self.use_ofc_integral and not self.control_zernikes:
            ki_correction = self.ki[self.indices] * np.sum(self.integral_terms, axis=0)
            tweak -= ki_correction
            self.integral_terms.append(tweak)
        return tweak

    def extractOpenLoopReproduction(self, table):
        """
        Build the open-loop reproduction (OLR) Zernike sequence from a
        nightly observation table by removing the applied DOF trims.

        Parameters
        ----------
        table : pd.DataFrame
            Nightly AOS table loaded from parquet.  Must contain columns
            'seq', 'zk_deviation_R00/R04/R40/R44', 'dof_state',
            'rotation_angle', and 'band'.

        Returns
        -------
        tuple : (seqs, z_measured, z_opens, rots, filters)
            seqs       : list of int  – sequence numbers without NaN Zernikes
            z_measured : list of np.ndarray (4, 23) – raw measured Zernikes
            z_opens    : list of np.ndarray (4, 23) – open-loop reproduction
            rots       : list of float – rotation angles
            filters    : list of str  – band names
        """
        z_measured = []
        z_opens = []
        seqs = []
        rots = []
        filters = []
        for seq_num in range(self.seq_start, self.seq_end):
            this_table = table[table['seq'] == seq_num]
            zernikes = np.zeros([4, 23])
            zernikes[0, :] = this_table['zk_deviation_R00'].values[0]
            zernikes[1, :] = this_table['zk_deviation_R04'].values[0]
            zernikes[2, :] = this_table['zk_deviation_R40'].values[0]
            zernikes[3, :] = this_table['zk_deviation_R44'].values[0]
            if np.isnan(zernikes).any():
                print(f"{seq_num} has NaNs — skipping")
                continue
            z_measured.append(zernikes)
            # Remove applied trim to get open-loop Zernikes
            trim = this_table['dof_state'].values[0][self.indices]
            z_open = self.applyTrim(zernikes, trim, subtract=False)
            z_opens.append(z_open)
            seqs.append(seq_num)
            rots.append(this_table['rotation_angle'].values[0])
            filters.append(this_table['band'].values[0].split('_')[0])
        return seqs, z_measured, z_opens, rots, filters

    def runPIDStep(self, olr_zernikes, trim, n, stored_tweak,
                   rotation_angle=0.0, filter_name='i',
                   subtract_intrinsics=False):
        """
        Execute a single step of the PID simulation loop.

        Behaviour depends on self.discard_intermediates and
        self.smith_corrector.

        Parameters
        ----------
        olr_zernikes : np.ndarray, shape (4, 23)
            Open-loop reproduction Zernikes for this exposure.
        trim : np.ndarray, shape (22,)
            Current accumulated DOF trim.
        n : int
            Step index (0-based).
        stored_tweak : np.ndarray or deque
            When discard_intermediates=True: np.ndarray(22,) holding the
            pending tweak.
            When discard_intermediates=False: deque of length n_correct
            holding the recent tweak history.
        rotation_angle : float
            Instrument rotator angle in degrees.
        filter_name : str
            Photometric band name.
        subtract_intrinsics : bool
            Passed through to getTweak.

        Returns
        -------
        tuple : (sim_zernikes, stored_tweak, trim)
            sim_zernikes : np.ndarray (4, 23) – simulated Zernikes after trim
            stored_tweak : updated tweak storage (same type as input)
            trim         : updated accumulated trim
        """
        if self.discard_intermediates:
            if n % self.n_correct == 0:
                trim += stored_tweak
                sim_zernikes = self.applyTrim(olr_zernikes, trim)
                stored_tweak = self.getTweak(
                    sim_zernikes,
                    rotation_angle=rotation_angle,
                    filter_name=filter_name,
                    subtract_intrinsics=subtract_intrinsics,
                )
            else:
                sim_zernikes = self.applyTrim(olr_zernikes, trim)
        else:
            trim += stored_tweak[0]
            sim_zernikes = self.applyTrim(olr_zernikes, trim)
            input_zernikes = copy.deepcopy(sim_zernikes)
            if self.smith_corrector:
                mod_sim_zernikes = self.sens_mat @ (stored_tweak[-1] - stored_tweak[-2])
                mod_sim_zernikes = mod_sim_zernikes.reshape(4, 21)
                # Zero out Z20 and Z21
                mod_sim_zernikes = np.insert(mod_sim_zernikes, obj=16, values=0, axis=1)
                mod_sim_zernikes = np.insert(mod_sim_zernikes, obj=16, values=0, axis=1)
                input_zernikes += mod_sim_zernikes
            stored_tweak.append(self.getTweak(
                input_zernikes,
                rotation_angle=rotation_angle,
                filter_name=filter_name,
                subtract_intrinsics=subtract_intrinsics,
            ))
        return sim_zernikes, stored_tweak, trim

    # ─────────────────────────────────────────────────────────────────────
    # Top-level run methods
    # ─────────────────────────────────────────────────────────────────────

    def buildOpenLoopReproduction(self, table):
        """
        Load the open-loop reproduction from the nightly table and store
        the results as instance attributes.

        Parameters
        ----------
        table : pd.DataFrame
            Nightly AOS table loaded from parquet.

        Side effects
        ------------
        Sets self.seqs, self.z_measured, self.z_opens, self.rots,
        self.filters.
        """
        (self.seqs, self.z_measured,
         self.z_opens, self.rots, self.filters) = self.extractOpenLoopReproduction(table)
        self.z_measured = np.array(self.z_measured)
        print(f"Built OLR: {len(self.seqs)} sequences from "
              f"{self.seq_start} to {self.seq_end}")

    def runSimulation(self):
        """
        Run the full PID simulation over all open-loop reproduction
        exposures and store the results as instance attributes.

        Requires buildOpenLoopReproduction to have been called first.

        Side effects
        ------------
        Sets self.new_zernikes, self.integrals, self.tweaks, self.trims,
        self.these_seqs.
        """
        # Push PID gains into the OFC controller
        if self.use_ofc_integral:
            self.ofc.controller.kp = self.kp
            self.ofc.controller.ki = self.ki
        else:
            if self.control_zernikes:
                self.ofc.controller.kp = np.ones(50)                    
                self.ofc.controller.ki = np.zeros(50)
            else:
                self.ofc.controller.ki = np.zeros(50)
            self.buildIntegralTerms()
        self.ofc.controller.kd = self.kd
        self.ofc_data.max_integral = self.max_integral
        self.ofc.controller.reset_history()
        
        trim = np.zeros(22)
        if self.discard_intermediates:
            stored_tweak = np.zeros(22)
        else:
            stored_tweak = deque(maxlen=self.n_correct)
            for _ in range(self.n_correct):
                stored_tweak.append(np.zeros(22))

        new_zernikes = []
        integrals = []
        tweaks = []
        trims = []
        these_seqs = []

        for n, olr_zernikes in enumerate(self.z_opens):
            sim_zernikes, stored_tweak, trim = self.runPIDStep(
                olr_zernikes, trim, n, stored_tweak,
                rotation_angle=self.rots[n],
                filter_name=self.filters[n],
                subtract_intrinsics=False,
            )
            trims.append(copy.deepcopy(trim))
            tweaks.append(stored_tweak[-1])
            new_zernikes.append(sim_zernikes)
            if self.use_ofc_integral:
                integrals.append(self.ofc.controller.integral)
            else:
                integrals.append(np.sum(self.integral_terms, axis=0))
            these_seqs.append(self.seqs[n])

        self.new_zernikes = np.array(new_zernikes)
        self.integrals = np.array(integrals)
        self.tweaks = np.array(tweaks)
        self.trims = np.array(trims)
        self.these_seqs = these_seqs

    # ─────────────────────────────────────────────────────────────────────
    # Plotting methods
    # ─────────────────────────────────────────────────────────────────────

    def plotPID(self, sub_title, title, plot_measured=False, save_fig=False):
        """
        Simple summary plot: mean Zernike residuals vs. sequence number
        for each Zernike group.

        Parameters
        ----------
        sub_title : str
            Descriptive subtitle added below the main figure title.
        title : str
            Base filename (without extension) used when saving the figure.
        plot_measured : bool
            If True, overlay the raw measured Zernikes for comparison.
        """
        fig = plt.figure(figsize=(10, 10))
        gs = gridspec.GridSpec(
            nrows=6, ncols=1,
            height_ratios=[1] * 6,
            hspace=0.0,
            wspace=0.26,
            top=0.95,
            bottom=0.05,
        )
        axes = [fig.add_subplot(gs[i, 0]) for i in range(6)]
        for id_group, (ax, zk_group) in enumerate(zip(axes, self.zk_groups)):
            zk_labels = self.zk_group_labels[id_group]
            ymin = -1.0
            ymax = 1.0
            for zk_idx, i in enumerate(zk_group):
                if len(zk_group) == 1:
                    zk_label = ''
                else:
                    zk_label = zk_labels.split('/')[zk_idx].strip()
                vals = np.mean(self.new_zernikes[:, :, i], axis=1)
                color = 'tab:blue' if zk_idx == 0 else 'tab:orange' if zk_idx == 1 else None
                ax.scatter(self.these_seqs, vals, s=20, color=color, label=zk_label)
                if plot_measured:
                    vals_2 = np.mean(self.z_measured[:, :, i], axis=1)
                    meas_color = 'red' if zk_idx == 0 else 'green' if zk_idx == 1 else None
                    ax.scatter(self.these_seqs, vals_2, s=20, color=meas_color,
                               marker='x', label="Meas_" + zk_label)
            ax.set_ylabel(f"{self.zk_group_labels[id_group]}\n[um]")
            if self.zk_group_labels[id_group] == 'Z4':
                ymin = -1
                ymax = 1
                ax.axhline(-0.15, ls='--', color='green')
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            ax.set_ylim(ymin, ymax)
            ax.set_xlabel("Sequence number")
            ax.legend(bbox_to_anchor=(-0.20, 0.5), loc='upper left',
                      ncol=1, markerscale=2, frameon=False)
        my_suptitle = fig.suptitle(
            f"Simulated PID loop start={self.seq_start}\n{sub_title}",
            y=1.02, fontsize=18,
        )
        if save_fig:
            fig.savefig(
            title,
            bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle],
        )

    def bigPlotPID1(self, sub_title, title, plot_measured=False, save_fig=False,):
        """
        Comprehensive diagnostic plot showing Zernikes at all four corner
        detectors (top panel) plus hexapod/mirror DOF trims and vmodes
        (bottom panel).

        Parameters
        ----------
        sub_title : str
            Descriptive subtitle added below the main figure title.
        title : str
            Base filename (without extension) used when saving the figure.
        plot_measured : bool
            If True, overlay the raw measured Zernikes on the mean column.
        """
        fig = plt.figure(figsize=(39, 19))

        gs_top = gridspec.GridSpec(
            nrows=6, ncols=6,
            width_ratios=[1] * 6,
            height_ratios=[1] * 6,
            hspace=0.0, wspace=0.38,
            top=0.97, bottom=0.54,
        )
        gs_bot = gridspec.GridSpec(
            nrows=6, ncols=6,
            width_ratios=[1] * 6,
            height_ratios=[1] * 6,
            hspace=0.0, wspace=0.38,
            top=0.46, bottom=0.05,
        )

        z_names = ['Mean', 'R00', 'R04', 'R40', 'R44']
        for j in range(5):
            axes = [fig.add_subplot(gs_top[i, j]) for i in range(6)]
            for id_group, (ax, zk_group) in enumerate(zip(axes, self.zk_groups)):
                zk_labels = self.zk_group_labels[id_group]
                ymin = -1.0
                ymax = 1.0
                for zk_idx, i in enumerate(zk_group):
                    if len(zk_group) == 1:
                        zk_label = ''
                    else:
                        zk_label = zk_labels.split('/')[zk_idx].strip()
                    vals = (np.mean(self.new_zernikes[:, :, i], axis=1)
                            if j == 0 else self.new_zernikes[:, j - 1, i])
                    color = 'tab:blue' if zk_idx == 0 else 'tab:orange' if zk_idx == 1 else None
                    ax.scatter(self.these_seqs, vals, s=20, color=color, label=zk_label)
                    if plot_measured and j == 0:
                        vals_2 = np.mean(self.z_measured[:, :, i], axis=1)
                        meas_color = 'red' if zk_idx == 0 else 'green' if zk_idx == 1 else None
                        ax.scatter(self.these_seqs, vals_2, s=20, color=meas_color,
                                   marker='x', label="Meas_" + zk_label)
                ax.set_ylabel(f"{self.zk_group_labels[id_group]}\n[um]")
                if self.zk_group_labels[id_group] == 'Z4':
                    ymin = -1
                    ymax = 1
                    ax.axhline(-0.15, ls='--', color='green')
                ax.grid(True, alpha=0.5)
                ax.tick_params(direction="in")
                ax.set_ylim(ymin, ymax)
                ax.set_xlabel("Sequence number")
                if id_group == 0:
                    ax.set_title(z_names[j])
                ax.legend(bbox_to_anchor=(-0.20, 0.5), loc='upper left',
                          ncol=1, markerscale=2, frameon=False)

        # Hexapod DOF trims
        axes = [fig.add_subplot(gs_bot[i, 0]) for i in range(6)]
        for id_group, (ax, dof_group) in enumerate(zip(axes, self.groups)):
            for i in dof_group:
                vals = self.trims[:, i]
                plot_vals = 3600.0 * vals if i in [3, 4, 8, 9] else vals
                ax.scatter(self.these_seqs, plot_vals, label=f"{self.labels[i]}", s=3)
            ax.set_ylabel(self.group_labels[id_group])
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            leg = ax.legend(bbox_to_anchor=(1.28, 0.5), loc='center right', markerscale=3)
            leg.get_frame().set_linewidth(0)
            leg.get_frame().set_edgecolor("none")
            leg.get_frame().set_facecolor("none")
        for ax in axes[:-1]:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes[0].set_title('Hexapods State (Trim only)')
        axes[-1].set_xlabel("Sequence Number")

        # Mirror DOF trims
        axes1 = [fig.add_subplot(gs_bot[i, 1]) for i in range(6)]
        axes2 = [fig.add_subplot(gs_bot[i, 2]) for i in range(6)]
        mirror_axes = axes1 + axes2
        mirror_indices = list(range(10, 17)) + list(range(30, 35))
        for n, i in enumerate(mirror_indices):
            mirror_axes[n].scatter(self.these_seqs, self.trims[:, n], s=11)
            mirror_axes[n].set_ylabel(self.all_labels[i])
        for ax in mirror_axes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes1[0].set_title('Mirror DOFs (Trim only)')
        axes1[5].tick_params(labelbottom=True)
        axes1[5].set_xlabel("Sequence Number")
        axes2[0].set_title('Mirror DOFs (Trim only)')
        axes2[5].tick_params(labelbottom=True)
        axes2[5].set_xlabel("Sequence Number")

        # Vmodes
        vmodes = []
        full_indices = list(range(0, 17)) + list(range(30, 35))
        for i in range(self.trims.shape[0]):
            this_trim = np.zeros(50)
            this_trim[full_indices] = self.trims[i, :]
            vmodes.append(self.state_estimator.get_vmodes_from_dofs(this_trim))
        vmodes = np.array(vmodes)

        vaxes1 = [fig.add_subplot(gs_bot[i, 3]) for i in range(6)]
        vaxes2 = [fig.add_subplot(gs_bot[i, 4]) for i in range(6)]
        vaxes = vaxes1 + vaxes2
        for i in range(12):
            vaxes[i].scatter(self.these_seqs, vmodes[:, i], s=11)
            vaxes[i].set_ylabel(self.vmode_labels[i])
        for ax in vaxes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        vaxes1[0].set_title('Vmodes')
        vaxes1[5].tick_params(labelbottom=True)
        vaxes1[5].set_xlabel("Sequence Number")
        vaxes2[0].set_title('Vmodes')
        vaxes2[5].tick_params(labelbottom=True)
        vaxes2[5].set_xlabel("Sequence Number")

        my_suptitle = fig.suptitle(
            f"Simulated PID loop start={self.seq_start}\n{sub_title}",
            y=1.02, fontsize=18,
        )
        if save_fig:
            fig.savefig(
            title,
            bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle],
        )

    def bigPlotPID2(self, sub_title, title, plot_measured=False, save_fig=False):
        """
        Alternative diagnostic plot showing mean Zernike residuals and
        integral terms (top panel) plus hexapod/mirror DOF trims and
        vmodes (bottom panel).

        Parameters
        ----------
        sub_title : str
            Descriptive subtitle added below the main figure title.
        title : str
            Base filename (without extension) used when saving the figure.
        plot_measured : bool
            If True, overlay the raw measured Zernikes for comparison.
        """
        fig = plt.figure(figsize=(39, 19))

        gs_top = gridspec.GridSpec(
            nrows=6, ncols=6,
            width_ratios=[1] * 6,
            height_ratios=[1] * 6,
            hspace=0.0, wspace=0.38,
            top=0.97, bottom=0.54,
        )
        gs_bot = gridspec.GridSpec(
            nrows=6, ncols=6,
            width_ratios=[1] * 6,
            height_ratios=[1] * 6,
            hspace=0.0, wspace=0.38,
            top=0.46, bottom=0.05,
        )

        # Zernike column
        axes = [fig.add_subplot(gs_top[i, 0]) for i in range(6)]
        axes[0].set_title("Zernikes")
        for id_group, (ax, zk_group) in enumerate(zip(axes, self.zk_groups)):
            zk_labels = self.zk_group_labels[id_group]
            ymin = -1.0
            ymax = 1.0
            for zk_idx, i in enumerate(zk_group):
                if len(zk_group) == 1:
                    zk_label = ''
                else:
                    zk_label = zk_labels.split('/')[zk_idx].strip()
                vals = np.mean(self.new_zernikes[:, :, i], axis=1)
                color = 'tab:blue' if zk_idx == 0 else 'tab:orange' if zk_idx == 1 else None
                ax.scatter(self.these_seqs, vals, s=20, color=color, label=zk_label)
                if plot_measured:
                    vals_2 = np.mean(self.z_measured[:, :, i], axis=1)
                    meas_color = 'red' if zk_idx == 0 else 'green' if zk_idx == 1 else None
                    ax.scatter(self.these_seqs, vals_2, s=20, color=meas_color,
                               marker='x', label="Meas_" + zk_label)
            ax.set_ylabel(f"{self.zk_group_labels[id_group]}\n[um]")
            if self.zk_group_labels[id_group] == 'Z4':
                ymin = -1
                ymax = 1
                ax.axhline(-0.15, ls='--', color='green')
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            ax.set_ylim(ymin, ymax)
            ax.set_xlabel("Sequence number")
            ax.legend(bbox_to_anchor=(-0.20, 0.5), loc='upper left',
                      ncol=1, markerscale=2, frameon=False)

        # Integral term columns
        integral_index = 0
        int_indices = list(range(0, 17)) + list(range(30, 35))
        for j in range(1, 5):
            int_axes = [fig.add_subplot(gs_top[i, j]) for i in range(6)]
            int_axes[0].set_title("Integral terms")
            for ax in int_axes:
                if integral_index > 21:
                    ax.set_visible(False)
                    continue
                vals = self.integrals[:, integral_index]
                int_label = self.all_labels[int_indices[integral_index]]
                color = 'tab:blue'
                ax.scatter(self.these_seqs, vals, s=20, color=color)
                ax.set_ylabel(int_label)
                ax.grid(True, alpha=0.5)
                ax.tick_params(direction="in")
                ax.set_xlabel("Sequence number")
                integral_index += 1

        # Hexapod DOF trims
        axes = [fig.add_subplot(gs_bot[i, 0]) for i in range(6)]
        for id_group, (ax, dof_group) in enumerate(zip(axes, self.groups)):
            for i in dof_group:
                vals = self.trims[:, i]
                plot_vals = 3600.0 * vals if i in [3, 4, 8, 9] else vals
                ax.scatter(self.these_seqs, plot_vals, label=f"{self.labels[i]}", s=3)
            ax.set_ylabel(self.group_labels[id_group])
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            leg = ax.legend(bbox_to_anchor=(1.28, 0.5), loc='center right', markerscale=3)
            leg.get_frame().set_linewidth(0)
            leg.get_frame().set_edgecolor("none")
            leg.get_frame().set_facecolor("none")
        for ax in axes[:-1]:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes[0].set_title('Hexapods State (Trim only)')
        axes[-1].set_xlabel("Sequence Number")

        # Mirror DOF trims
        axes1 = [fig.add_subplot(gs_bot[i, 1]) for i in range(6)]
        axes2 = [fig.add_subplot(gs_bot[i, 2]) for i in range(6)]
        mirror_axes = axes1 + axes2
        mirror_indices = list(range(10, 17)) + list(range(30, 35))
        for n, i in enumerate(mirror_indices):
            mirror_axes[n].scatter(self.these_seqs, self.trims[:, n], s=11)
            mirror_axes[n].set_ylabel(self.all_labels[i])
        for ax in mirror_axes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes1[0].set_title('Mirror DOFs (Trim only)')
        axes1[5].tick_params(labelbottom=True)
        axes1[5].set_xlabel("Sequence Number")
        axes2[0].set_title('Mirror DOFs (Trim only)')
        axes2[5].tick_params(labelbottom=True)
        axes2[5].set_xlabel("Sequence Number")

        # Vmodes
        vmodes = []
        full_indices = list(range(0, 17)) + list(range(30, 35))
        for i in range(self.trims.shape[0]):
            this_trim = np.zeros(50)
            this_trim[full_indices] = self.trims[i, :]
            vmodes.append(self.state_estimator.get_vmodes_from_dofs(this_trim))
        vmodes = np.array(vmodes)

        vaxes1 = [fig.add_subplot(gs_bot[i, 3]) for i in range(6)]
        vaxes2 = [fig.add_subplot(gs_bot[i, 4]) for i in range(6)]
        vaxes = vaxes1 + vaxes2
        for i in range(12):
            vaxes[i].scatter(self.these_seqs, vmodes[:, i], s=11)
            vaxes[i].set_ylabel(self.vmode_labels[i])
        for ax in vaxes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        vaxes1[0].set_title('Vmodes')
        vaxes1[5].tick_params(labelbottom=True)
        vaxes1[5].set_xlabel("Sequence Number")
        vaxes2[0].set_title('Vmodes')
        vaxes2[5].tick_params(labelbottom=True)
        vaxes2[5].set_xlabel("Sequence Number")

        my_suptitle = fig.suptitle(
            f"Simulated PID loop start={self.seq_start}\n{sub_title}",
            y=1.02, fontsize=18,
        )
        if save_fig:
            fig.savefig(
            title,
            bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle],
        )


## Build OFC, state estimator and sensitivity matrix
You will need to `git clone ts_config_mttcs` and replace the config path below.

In [ ]:
ofc_data = OFCData(
    name="lsst",
    config_dir="/home/c/cslage/WORK/ts_config_mttcs/MTAOS/ofc",
)

ofc = OFC(ofc_data=ofc_data)

new_comp_dof_idx = dict(
    m2HexPos=np.ones(5, dtype=bool),
    camHexPos=np.ones(5, dtype=bool),
    M1M3Bend=np.ones(20, dtype=bool),
    M2Bend=np.ones(20, dtype=bool),
)
new_comp_dof_idx["M1M3Bend"][7:] = False
new_comp_dof_idx["M2Bend"][5:] = False

ofc.set_truncation_index(12)
ofc_data.zn_selected = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26])
ofc.ofc_data.comp_dof_idx = new_comp_dof_idx
ofc.controller.reset_history()
ofc.state_estimator.refresh_from_ofc_data()

corner_detnames = ["R00_SW0", "R04_SW0", "R40_SW0", "R44_SW0"]
field_angles_CCS = [ofc_data.sample_points[det] for det in corner_detnames]

state_estimator = StateEstimator(ofc_data)
sens_mat = state_estimator.get_sensitivity_matrix(field_angles_CCS, 0.0)

## Set PID gains and instantiate the simulation
Edit the gain arrays and parameters below to configure the run.

In [ ]:
# DOF index: collapses the full 50-DOF space down to the 22 DOFs we are using.
# If control_vmodes=True the _used arrays should have 12 components.
# If control_vmodes=False they should have 22 components.
indices = list(range(0, 17)) + list(range(30, 35))

"""

# ── Starting point for control_vmodes=False ───────────────────────────
kp = np.zeros(50)
kp_used = 0.3 * np.ones(22)
kp[indices] = kp_used

ki = np.zeros(50)
ki_used = 0.0 * np.ones(22)
ki[indices] = ki_used

kd = np.zeros(50)



# ── Starting point for control_vmodes=True ────────────────────────────
kp = np.zeros(50)
kp_used = np.array([0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.05, 0.3, 0.3])
kp[0:12] = kp_used
ki = np.zeros(50)
ki_used = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
ki[0:12] = ki_used
kd = np.zeros(50)

"""

# Starting point when control_zernikes=True
kd = np.zeros(50)
kp = 0.3 * np.ones([4,23])
ki = np.zeros([4,23])
for i in range(4):
    kp[i,0] = 1.0
    #kp[i,7] = 0.0


# Integral term options. Only used when use_ofc_integral=True
# ── Integral clamp (max absolute value the integral term can reach) ───
max_integral = np.zeros(50)
max_integral_used = 5000.0 * np.ones(22)
max_integral[indices] = max_integral_used

## Notes on Ki options:
(1) use_ofc_integral=True\
Integral term for each DoF will be kept inside ofc and trimmed using max_integral.\
\
(2) use_ofc_integral=False and control_zernikes=False\
Integral term for each DoF will be kept in the notebook.\
No trimming, each DoF will keep the integral for n_integral terms.\
\
(3) use_ofc_integral=False and control_zernikes=True\
Kp and Ki terms will be applied in Zernike space.\
Each Zernike coefficient will keep the integral for n_integral terms.\

In [ ]:
sim = PIDLoopSimulation(
    ofc=ofc,
    ofc_data=ofc_data,
    state_estimator=state_estimator,
    sens_mat=sens_mat,
    kp=kp,
    ki=ki,
    kd=kd,
    day_obs=20260420,
    seq_start=97,
    seq_end=400,
    n_correct=3,
    discard_intermediates=True,
    smith_corrector=False,
    control_vmodes=False,
    use_ofc_integral=False,
    max_integral=max_integral,
    control_zernikes=True,
    n_integrals=10   
)

## Load the nightly parquet table
This parquet file must be extracted at the summit — USDF software is behind!

The parquet file was created with:
https://github.com/lsst-sitcom/ts_aos_analysis/blob/tickets/DM-54406/notebooks/nightly_report/nightly_report_ts_version.ipynb

In [ ]:
parquet_file = f"/home/c/cslage/u/MTAOS/times_square_reports/nightly_aos_table_{sim.day_obs}_summit.parquet"
table = pd.read_parquet(parquet_file)
print(f'Loaded {parquet_file}: {len(table)} rows')
print(f'Columns: {sorted(table.columns.tolist())}')

## Build the open-loop reproduction

In [ ]:
sim.buildOpenLoopReproduction(table)

## Run the simulation
Integral terms, trims, and tweaks are stored on the `sim` object for later plotting.

In [ ]:
sim.runSimulation()

## Plot the results
Three plotting options are available:
- `sim.plotPID(...)` — mean Zernike residuals only
- `sim.bigPlotPID1(...)` — Zernikes at all 4 corners + DOFs + vmodes
- `sim.bigPlotPID2(...)` — integral terms + DOFs + vmodes

In [ ]:
sub = f"{sim.day_obs}, Kp=0.3,Kp(Z4)=1.0, Ki=0.0, discard intermediates, Vmodes, control Zernikes"
sim.plotPID(sub, f"/home/c/cslage/u/MTAOS/pid_output/PID_Simulator_ZControl_{sim.day_obs}.png", plot_measured=True, save_fig=True)
# sim.plotPID(sub, f"PID_Simulator_{sim.day_obs}", plot_measured=True)

In [ ]:
sub = f"{sim.day_obs}, Kp=0.3, Ki=0.0, discard intermediates, Vmodes"
sim.bigPlotPID1(sub, f"PID_Simulator_Test_{sim.day_obs}")
# sim.bigPlotPID1(sub, f"...", plot_measured=True)

In [ ]:
sub = f"{sim.day_obs}, Kp=0.3,Kp(10)=0.05, Ki=0.0,Ki(10)=0.5, discard intermediates, Vmodes, integrate_dofs, local_integrals"
sim.bigPlotPID2(sub, f"/home/c/cslage/u/MTAOS/pid_output/PID_Simulator_V5_{sim.day_obs}.png", save_fig=True)
# sim.bigPlotPID2(sub, f"...", plot_measured=True)